In [ ]:
import sys
sys.path.insert(0, '/home/phil/Dev/Python/Single Cell/scvelo')
sys.path.insert(0, '/home/phil/Dev/Python/Single Cell/scanpy')
%load_ext autoreload
%autoreload 2

In [ ]:
import tarfile
from tempfile import gettempdir
from urllib.request import urlretrieve
from pathlib import Path
from string import ascii_lowercase

import matplotlib.pyplot as plt
#import tikzplotlib
import numpy as np
import scipy.io
import pandas as pd
import scanpy as sc
import scvelo as scv
from anndata import AnnData
from subprocess import run, CalledProcessError
%matplotlib inline

download data

In [ ]:
tmp_path = Path(gettempdir()) / "dim-red"
tmp_path.mkdir(exist_ok=True)

if not (tar_path := tmp_path / "destiny.tar.gz").is_file():
    urlretrieve("https://bioconductor.org/packages/3.10/bioc/src/contrib/destiny_3.0.1.tar.gz", tar_path)

if not (rda_path := tmp_path / "guo_norm.rda").is_file():
    with tarfile.open(tar_path) as tf:
        reader = tf.extractfile("destiny/data/guo_norm.rda")
        rda_path.write_bytes(reader.read())

exprs_path = tmp_path / "exprs.mtx"
obs_path = tmp_path / "obs.feather"
var_path = tmp_path / "var.feather"

load data

In [ ]:
script = f"""\
load({str(rda_path)!r})
Matrix::writeMM(Matrix::Matrix(t(guo_norm@assayData$exprs), sparse=TRUE), {str(exprs_path)!r})
feather::write_feather(guo_norm@phenoData@data, {str(obs_path)!r})
feather::write_feather(data.frame(var_names=rownames(guo_norm@assayData$exprs)), {str(var_path)!r})
""".replace("\n", ";")
p = run(["R", "--slave", "-e", script], capture_output=True)
if p.stdout: print(p.stdout.decode("latin-1"))
if p.stderr: print(p.stderr.decode("latin-1"), file=sys.stderr)

In [ ]:
adata = AnnData(
    scipy.io.mmread(str(exprs_path)).A,
    pd.read_feather(obs_path),
    pd.read_feather(var_path).set_index("var_names"),
)
#adata.obs["num_cells"] = pd.Categorical(adata.obs["num_cells"])
adata

analyze

In [ ]:
#sc.pp.log1p(adata)
sc.pp.neighbors(adata, 20, random_state=0)
sc.tl.pca(adata, random_state=0)
sc.tl.diffmap(adata)
sc.tl.umap(adata, random_state=0)

plot

In [ ]:
#if plt.matplotlib.font_manager.findfont("TeX Gyre Pagella").endswith("DejaVuSans.ttf"):
    #raise RuntimeError("TTF:", sorted({f.name for f in plt.matplotlib.font_manager.fontManager.ttflist}))
    #print("AFM:", sorted({f.name for f in plt.matplotlib.font_manager.fontManager.afmlist}))

In [ ]:
plt.style.use("seaborn-paper")
sc.settings.set_figure_params(fontsize=10)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.serif'] = ['Palatino', 'Palatino Linotype', 'TeX Gyre Pagella']
plt.rcParams['lines.linewidth'] = .5

fig = plt.figure(figsize=(6.5, 1.9))
gs = fig.add_gridspec(1, 4)
kws = [
    dict(x="Gata4", y="Nanog"),
    dict(basis="pca"),
    dict(basis="diffmap"),
    dict(basis="umap"),
]
# hack: https://github.com/theislab/scvelo/issues/150
adata.uns['num_cells_colors'] = list(map(plt.cm.colors.to_hex, plt.cm.viridis(np.linspace(0, 1, 6))))
for letter, kw, ax in zip(ascii_lowercase, kws, map(fig.add_subplot, gs)):
    scv.pl.scatter(adata, **kw, color='num_cells', palette='viridis', frameon=True, title="", ax=ax, show=False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(-0.15, 1, letter, transform=ax.transAxes, size=16, weight='bold')

#plt.xlabel("time (s)")
#plt.ylabel("Voltage (mV)")
#plt.title("Simple plot $\\frac{\\alpha}{2}$")
#plt.grid(True)
 
#tikzplotlib.save("dim-red.tikz", flavor="context")
fig.tight_layout()
fig.savefig("dim-red.pdf")